# Configurar la ruta de Power BI

Ejecute la celda con Power BI Desktop cerrado, antes de abrir `powerbi/Caso_Adopcion_Digital.pbip`. El modelo utiliza los seis CSV de `output/`.

Inicie Jupyter en la carpeta principal del proyecto o en `notebooks/`. Si se inició en otra carpeta, indique la carpeta principal cuando se solicite. El cuaderno cambia únicamente el parámetro `pArchivo` del PBIP.


In [ ]:
from pathlib import Path
import os
import re

base = Path.cwd().resolve()
if base.name.lower() == 'notebooks':
    base = base.parent
if not (base / 'powerbi' / 'Caso_Adopcion_Digital.pbip').is_file():
    base = Path(input('Carpeta principal del proyecto: ').strip().strip('\"')).expanduser().resolve()

pbip = base / 'powerbi' / 'Caso_Adopcion_Digital.pbip'
modelo = base / 'powerbi' / 'Caso_Adopcion_Digital.SemanticModel' / 'definition' / 'expressions.tmdl'
archivos = ('FactClienteCorte.csv', 'FactMovimientos.csv', 'DimCliente.csv',
            'DimProducto.csv', 'DimTienda.csv', 'DimReglaAnalisis.csv')
if os.name != 'nt':
    raise OSError('La configuración de Power BI Desktop se realiza en Windows.')
if not pbip.is_file() or not modelo.is_file():
    raise FileNotFoundError('Seleccione la carpeta que contiene powerbi/.')
faltantes = [nombre for nombre in archivos if not (base / 'output' / nombre).is_file()]
if faltantes:
    raise FileNotFoundError('Faltan archivos en output/: ' + ', '.join(faltantes))

contenido = modelo.read_text(encoding='utf-8')
patron = r'(?m)^(expression\s+pArchivo\s*=\s*)"([^"]*)"(\s+meta\s+\[IsParameterQuery=true, Type="Text", IsParameterQueryRequired=true\])'
coincidencia = re.search(patron, contenido)
if not coincidencia:
    raise ValueError('No se encontró el parámetro pArchivo en el modelo.')
ruta = str((base / 'output').resolve())
nuevo = contenido[:coincidencia.start(2)] + ruta + contenido[coincidencia.end(2):]
if nuevo != contenido:
    modelo.write_text(nuevo, encoding='utf-8')
print('Ruta de Power BI configurada:', ruta)
